## In sample Study

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

from strategies.supertrend_ema_confirmation.strategy import (
    SupertrendEmaConfirmationStrategy as Strategy,
)

## Constants

In [ ]:
from pathlib import Path

data_storage_path = Path.cwd().parent / "data"
backtest_results_dir = Path.cwd().parent / "backtest_results"
backtest_results_dir_in_sample = backtest_results_dir / "in_sample"
reports_dir = Path.cwd().parent / "reports"
figures_dir = reports_dir / "figures"
time_frames = ["2h", "4h", "1d"]

## Study definition

Here we define our insample study. Our insample study is defined as follows:
- Rolling backtest windows with a train period of 365 days, a test period of 180 days and a gap of 30 days between the train and test periods.
- Universe consisting of BTC, ETH, ADA, SOL, and DOT trading against EUR on the BITVAVO market.
- Backtest engine set to VECTOR, we only want to use the vector engine for this study. The goal is to go through as many algorithms as possible and find the best performing one.
- Risk free rate set to 0.027 (2.7%).

In [ ]:
from datetime import datetime, timezone
from investing_algorithm_framework import generate_rolling_backtest_windows, Study, \
    BacktestEngine, Universe, show_study, WindowPart, StudySampleType

rolling_backtest_windows = generate_rolling_backtest_windows(
    start_date=datetime(2022, 1, 1, tzinfo=timezone.utc),
    end_date=datetime(2025, 12, 30, tzinfo=timezone.utc),
    train_days=365,
    test_days=180,
    gap_days=30,
    step_days=90,
)

study = Study(
    name="in_sample_param_sweep",
    description="In-sample parameter sweep over the in_sample_basket universe "
    "(BTC/ETH/ADA/SOL/DOT on BITVAVO/EUR, 2022-01 → 2025-12).",
    risk_free_rate=0.027,
    initial_capital=1000,
    sample_type=StudySampleType.IN_SAMPLE,
    universe=Universe(
        key="in_sample_basket",
        symbols=["BTC", "ETH", "ADA", "SOL", "DOT"],
        trading_symbol="EUR",
        market='BITVAVO'
    ),
    backtest_windows=rolling_backtest_windows,
    window_part=WindowPart.TEST,
    engines=[BacktestEngine.VECTOR]
)

show_study(study)

## Algorithm ID Generation

This is very important to ensure that the algorithm ID is unique for each parameter combination and trackable across different studies. You can use the framework provided function `generate_algorithm_id` to create a unique ID based on the algorithm name and its parameters. However, because we also provide some metadata 
to our algorithm variants we are going to use a custom function to generate the algorithm ID. This will allow us to include the metadata in the ID generation process.

In [ ]:
from investing_algorithm_framework import generate_algorithm_id

def custom_generate_algorithm_id(params: dict) -> str:
    """
    Custom function to generate a unique algorithm ID based on the algorithm name,
    its parameters, and additional metadata.
    """
    # We exlicitly generate an id from the stable subset of parameters
    # (the ones that don't #start with "_") and pass it to the strategy constructor.
    # This ensures that the same
    # strategy fingerprint is generated regardless of any additional metadata
    # we include in the variant for tracking purposes (like "_grid_profile"). By
    # filtering out keys that start with "_", we ensure that the
    # algorithm_id remains consistent across different runs and environments,
    # as it only depends on the core strategy parameters.
    # Drop ``_grid_profile`` and any other underscore-prefixed
    # metadata: those keys describe the *grid coordinate*, not the
    # strategy behaviour, and we want the lineage hash to be
    # invariant across notebook boundaries.
    strategy_params = {
        k: v for k, v in params.items() if not k.startswith("_")
    }

    return generate_algorithm_id(params=strategy_params)

In [ ]:
from itertools import product
from investing_algorithm_framework import create_markdown_table


# ══════════════════════════════════════════════════════════════════
#  BASE PARAMS — SuperTrend + EMA Confirmation Strategy
# ══════════════════════════════════════════════════════════════════
params = {
    # ── Timeframes ───────────────────────────────────────────────
    "ema_timeframe": "2h",
    "rsi_timeframe": "2h",

    # ── EMA settings ─────────────────────────────────────────────
    "ema_short_period": 50,
    "ema_long_period": 200,
    "ema_cross_lookback_window": 48,

    # ── RSI settings ─────────────────────────────────────────────
    "rsi_period": 14,
    "rsi_overbought_threshold": 75,
    "rsi_oversold_threshold": 30,

    # ── SuperTrend (primary trend filter) ────────────────────────
    "supertrend_atr_length": 10,
    "supertrend_factor": 3.0,
    "use_supertrend_filter": True,

    # ── Bollinger Bands (overextension guardrail) ────────────────
    "bollinger_period": 20,
    "bollinger_std_dev": 2.0,
    "use_bollinger_filter": True,

    # ── Risk management ──────────────────────────────────────────
    "stop_loss_percentage": 5.0,
    "take_profit_percentage": 10.0,
    "trailing_stop_loss": True,

    # ── Cooldown rules (in bars of ema_timeframe) ───────────────
    "reentry_cooldown_bars": 12,
    "portfolio_cooldown_bars": 2,

    # ── Short selling (off by default; the `shorting` grid
    #    dimension below flips this on for half the combos) ──────
    "enable_shorting": False,
    "cover_requires_current_trend": True,
    "cover_min_confirmation_bars": 3,
}


# ══════════════════════════════════════════════════════════════════
#  GRID PROFILES (target ≤ 100 combinations)
# ══════════════════════════════════════════════════════════════════

# ── Q0: Timeframe ────────────────────────────────────────────────
timeframe_profiles = {
    "2h": {"ema_timeframe": "2h", "rsi_timeframe": "2h"},
    "4h": {"ema_timeframe": "4h", "rsi_timeframe": "4h"},
}

# ── Q1: EMA period pairs ────────────────────────────────────────
ema_period_profiles = {
    "fast_21_100": {
        "ema_short_period": 21,
        "ema_long_period": 100,
        "ema_cross_lookback_window": 24,
    },
    "default_50_200": {
        "ema_short_period": 50,
        "ema_long_period": 200,
        "ema_cross_lookback_window": 48,
    },
}

# ── Q2: SuperTrend ───────────────────────────────────────────────
supertrend_profiles = {
    "responsive_10_2_5": {
        "supertrend_atr_length": 10,
        "supertrend_factor": 2.5,
    },
    "default_10_3_0": {
        "supertrend_atr_length": 10,
        "supertrend_factor": 3.0,
    },
}

# ── Q3: RSI thresholds ──────────────────────────────────────────
rsi_threshold_profiles = {
    "tight_70_30": {
        "rsi_overbought_threshold": 70,
        "rsi_oversold_threshold": 30,
    },
    "wide_75_25": {
        "rsi_overbought_threshold": 75,
        "rsi_oversold_threshold": 25,
    },
}

# ── Q4: Risk management ─────────────────────────────────────────
risk_profiles = {
    "tight_sl3_tp8": {
        "stop_loss_percentage": 3.0,
        "take_profit_percentage": 8.0,
        "trailing_stop_loss": True,
    },
    "default_sl5_tp10": {
        "stop_loss_percentage": 5.0,
        "take_profit_percentage": 10.0,
        "trailing_stop_loss": True,
    },
}

# ── Q5: Cooldown rules ──────────────────────────────────────────
cooldown_profiles = {
    "none": {
        "reentry_cooldown_bars": 0,
        "portfolio_cooldown_bars": 0,
    },
    "light": {
        "reentry_cooldown_bars": 6,
        "portfolio_cooldown_bars": 1,
    },
    "strict": {
        "reentry_cooldown_bars": 12,
        "portfolio_cooldown_bars": 2,
    },
}

# ── Q6: Short selling ───────────────────────────────────────────
# Toggles the SHORT/COVER signal branch on the strategy. The
# ``long_short_strict`` profile pairs ``enable_shorting=True`` with
# the tightened cover gates (current-trend SuperTrend + sustained
# EMA dominance) added in the strategy to avoid premature covers
# on minor counter-trend bounces.
shorting_profiles = {
    "long_only": {
        "enable_shorting": False,
    },
    "long_short_strict": {
        "enable_shorting": True,
        "cover_requires_current_trend": True,
        "cover_min_confirmation_bars": 3,
    },
}


# ══════════════════════════════════════════════════════════════════
#  BUILD PARAMETER VARIATIONS (cartesian product of profiles)
# ══════════════════════════════════════════════════════════════════
# Total: 2 × 2 × 2 × 2 × 3 × 2 = 96 combinations
# (SuperTrend held at its default; explored separately in a dedicated
#  sweep to keep this grid under 100 while still covering shorting.)

grid_dimensions = {
    "timeframe":       timeframe_profiles,
    "ema_periods":     ema_period_profiles,
    "rsi_thresholds":  rsi_threshold_profiles,
    "risk":            risk_profiles,
    "cooldowns":       cooldown_profiles,
    "shorting":        shorting_profiles,
}

dim_names = list(grid_dimensions.keys())
dim_items = [list(grid_dimensions[d].items()) for d in dim_names]

param_variations = []
for combo in product(*dim_items):
    variant = dict(params)  # start from base params
    profile_tag = {}
    for dim_name, (profile_name, overrides) in zip(dim_names, combo):
        variant.update(overrides)
        profile_tag[dim_name] = profile_name
    # Store profile tags for post-hoc analysis
    variant["_grid_profile"] = profile_tag
    param_variations.append(variant)

assert len(param_variations) <= 100, (
    f"Grid has {len(param_variations)} combinations — exceeds 100 cap"
)


# ══════════════════════════════════════════════════════════════════
#  PRINT OVERVIEW
# ══════════════════════════════════════════════════════════════════

print(f"Total parameter combinations: {len(param_variations)}")

# ── Default params ───────────────────────────────────────────────
param_groups = {
    "Timeframes":       ["ema_timeframe", "rsi_timeframe"],
    "EMA":              ["ema_short_period", "ema_long_period", "ema_cross_lookback_window"],
    "RSI":              ["rsi_period", "rsi_overbought_threshold", "rsi_oversold_threshold"],
    "SuperTrend":       ["supertrend_atr_length", "supertrend_factor", "use_supertrend_filter"],
    "Bollinger":        ["bollinger_period", "bollinger_std_dev", "use_bollinger_filter"],
    "Risk management":  ["stop_loss_percentage", "take_profit_percentage", "trailing_stop_loss"],
    "Cooldowns":        ["reentry_cooldown_bars", "portfolio_cooldown_bars"],
    "Shorting":         ["enable_shorting", "cover_requires_current_trend", "cover_min_confirmation_bars"],
}

print(f"\n{'═'*70}")
print(f"  DEFAULT BASE PARAMS (before grid overrides)")
print(f"{'═'*70}")

for group_name, keys in param_groups.items():
    matching = {k: params[k] for k in keys if k in params}
    if not matching:
        continue
    print(f"\n  {group_name}:")
    for k, v in matching.items():
        print(f"    {k:35s} = {v}")

# ── Grid dimensions summary table ────────────────────────────────
print(f"\n{'═'*70}")
print(f"  GRID DIMENSIONS ({len(grid_dimensions)} dimensions)")
print(f"{'═'*70}\n")

grid_table = [
    {"Dimension": dim, "Profiles": ", ".join(grid_dimensions[dim].keys()), "Count": len(grid_dimensions[dim])}
    for dim in dim_names
]
print(create_markdown_table(grid_table))

for dim_name, profiles in grid_dimensions.items():
    print(f"\n{'─'*60}")
    print(f"  {dim_name} ({len(profiles)} profiles)")
    print(f"{'─'*60}")
    for profile_name, overrides in profiles.items():
        if overrides:
            param_str = ", ".join(f"{k}={v}" for k, v in overrides.items())
        else:
            param_str = "(no overrides — uses class defaults)"
        print(f"  {profile_name:20s} │ {param_str}")

## Strategy and Backtest Initialization

In this section we initialize a backtest for each parameter combination in our grid. We use the same in-sample universe and time period for all backtests, so that their results are directly comparable.
Also we generate a unique `algorithm_id` for each strategy based on its parameters, which allows us to track the lineage of each backtest and link it back to the original parameter combination. This willb
be uses as the unique fingerprint for each strategy variant, ensuring that we can trace back the performance of each backtest to its specific parameter settings.

In [ ]:
from investing_algorithm_framework import TimeUnit
from investing_algorithm_framework.domain import tqdm

# Map timeframe strings to (time_unit, interval) scheduling pairs.
TIMEFRAME_TO_SCHEDULE = {
    "2h":  (TimeUnit.HOUR, 2),
    "4h":  (TimeUnit.HOUR, 4),
    "1d":  (TimeUnit.DAY, 1),
}


def initialize_strategies(
    strategy_class,
    param_variations,
    symbols,
    market,
    trading_symbol="EUR",
    filter_fn=None
):
    """Initialize multiple SupertrendEmaConfirmationStrategy instances
    with different parameters from the grid search.
    """
    strategies = []

    for variant in tqdm(
        param_variations, desc="Initializing strategies", colour="green"
    ):
        tf_str = variant.get("ema_timeframe", "2h")
        time_unit, interval = TIMEFRAME_TO_SCHEDULE[tf_str]

        # Strip underscore-prefixed metadata keys before passing to strategy
        strategy_params = {k: v for k, v in variant.items() if not k.startswith("_")}

        strategy = strategy_class(
            algorithm_id=custom_generate_algorithm_id(params=variant),
            symbols=symbols,
            trading_symbol=trading_symbol,
            market=market,
            metadata={
                "params": variant,
                "time_unit": TimeUnit(time_unit).name,
                "interval": interval,
                "symbols": symbols,
                "market": market,
            },
            **strategy_params,
        )
        strategies.append(strategy)

    if filter_fn:
        strategies = [s for s in strategies if filter_fn(s)]

    return strategies

strategies = initialize_strategies(
    strategy_class=Strategy,
    param_variations=param_variations,
    symbols=study.universe.symbols,
    market=study.universe.market,
)

In [ ]:
import os
from investing_algorithm_framework import (
    rank_results,
    BacktestEvaluationFocus,
    Backtest,
    create_app,
    RESOURCE_DIRECTORY,
    DATA_DIRECTORY
)

# Configuration constants for progressive pruning. Forgiving early on
# (a single bad early window must not kill a strategy that recovers
# later) and strict once enough windows are accumulated to judge
# consistency.
MIN_TRADES_PER_WINDOW = 1            # per-window: must have traded
WARMUP_WINDOWS = 3                   # require ≥ N windows before strict checks
MIN_TRADE_WINDOW_RATIO = 0.5         # ≥ 50% of windows must have trades
MAX_UNPROFITABLE_WINDOW_RATIO = 0.5  # ≤ 50% of windows may be losers
MIN_SUMMARY_NET_GAIN = 0.0           # bundle's cross-window PnL > 0

# Engine used for in-sample sweep. v9.0 ``Backtest`` carries separate
# ``vector_summary`` and ``event_summary`` slots (with their own
# ``*_runs``) so any per-bundle summary access MUST go through
# ``backtest.get_summary(engine)`` — the legacy single
# ``backtest.backtest_summary`` attribute no longer exists.
ENGINE = "vector"


def window_filter_function(backtests, backtest_date_range, engine=ENGINE):
    """Progressive pruning using engine-scoped summary metrics.

    Two layers, each forgiving on its own but cumulatively decisive:

    1. **Per-window check** (against the *current* window's
       :class:`BacktestMetrics`) — drops runs that produced *zero*
       closed trades in this window. Anything that traded at least
       once is allowed through to the cross-window layer.
    2. **Per-bundle cross-window checks** (against
       ``backtest.get_summary(engine)``) — only kick in once the
       summary has aggregated at least ``WARMUP_WINDOWS`` windows.
       Below that threshold we keep everything (a single bad early
       window must not kill a strategy that recovers later). Once
       warm, we require ≥ 50% of windows traded, ≤ 50% losing
       windows, and overall positive PnL across the bundle.
    """
    print(
        f"\nFiltering {len(backtests)} backtests for window: "
        f"{backtest_date_range.name} (engine={engine})"
    )

    def progressive_filter(backtest_metrics, backtest: Backtest):
        # 1) Per-window check: must have traded at all in this window.
        if (
            backtest_metrics.number_of_trades_closed is None
            or backtest_metrics.number_of_trades_closed
            < MIN_TRADES_PER_WINDOW
        ):
            return False

        # 2) Per-bundle (cross-window) consistency checks. ``rank_results``
        #    already filters out bundles without a populated ``engine``
        #    slot, so ``get_summary(engine)`` will normally be non-None;
        #    we still guard defensively.
        summary = backtest.get_summary(engine, study=study)

        if (
            summary is None
            or summary.number_of_windows is None
            or summary.number_of_windows < WARMUP_WINDOWS
        ):
            # Warmup phase: not enough windows to judge consistency.
            # Be forgiving — keep the candidate so it gets a fair
            # chance to recover in later windows.
            return True

        # 2a) Activity coverage: enough windows actually traded.
        trade_window_ratio = (
            summary.number_of_windows_with_trades
            / summary.number_of_windows
        )
        if trade_window_ratio < MIN_TRADE_WINDOW_RATIO:
            return False

        # 2b) Loss coverage: too many losing windows ⇒ inconsistent.
        unprofitable_ratio = (
            (summary.number_of_windows
             - summary.number_of_profitable_windows)
            / summary.number_of_windows
        )
        if unprofitable_ratio > MAX_UNPROFITABLE_WINDOW_RATIO:
            return False

        # 2c) Aggregate PnL must be positive across the bundle.
        if (
            summary.total_net_gain is None
            or summary.total_net_gain <= MIN_SUMMARY_NET_GAIN
        ):
            return False

        return True

    ranked_backtests = rank_results(
        backtests.copy(),
        filter_fn=progressive_filter,
        focus=BacktestEvaluationFocus.BALANCED,
        backtest_date_range=backtest_date_range,
        engine=engine,
        study=study
    )

    print(f"Backtests after filtering: {len(ranked_backtests)}")
    return ranked_backtests

# Run backtests
app = create_app(config={RESOURCE_DIRECTORY: "./resources", DATA_DIRECTORY: data_storage_path})

# Vector backtest run on our defined study, with progressive pruning across the rolling windows.
backtests_in_sample = app.run_backtest(
    strategies=strategies,
    study=study,
    continue_on_error=False,
    use_checkpoints=True,
    backtest_storage_directory=backtest_results_dir,
    show_progress=True,
    n_workers=os.cpu_count() - 3,
    dynamic_position_sizing=True,
    window_filter_function=window_filter_function,
    iterative_summary_update=True,
)

## Ranking & Promoting Top Backtests

After the parameter sweep completes, we often have dozens (or hundreds) of backtest bundles on disk. Rather than decoding every `.iafbt` file to compare them, we use the **Tier-1 SQLite index** to rank in milliseconds, then promote a copy of the winners to a dedicated folder so notebook 04 can pick them up cleanly:

1. **`build_index()`** — scans the results directory and promotes every scalar metric from each bundle into a single SQLite table (one row per backtest). This only touches the lightweight msgpack header, so it scales to 10k+ bundles.
2. **`rank_index()`** — runs a pure-SQL sort/filter on that index. Here we rank with the BALANCED focus and require at least 5 closed trades, returning the top 20 instantly — no full bundle decode needed. Bundles already in `top_selection/` are excluded so that re-running this cell doesn't trigger a self-copy error.
3. **Promote winners to `top_selection/`** — we wipe and recreate `top_selection/`, then copy the top-N bundles into it. This guarantees the folder reflects *only* this run's winners (no leakage from prior sweeps) and matches what notebook 04 reads back. The original sweep results stay intact in `backtest_results/`.

In [ ]:
from pathlib import Path
from investing_algorithm_framework import BacktestEvaluationFocus, build_index, \
    rank_index, format_table, promote_backtests

backtest_results_dir = Path.cwd().parent / "backtest_results"

# 1. Build (or refresh) the Tier-1 SQLite index over the full sweep.
#    exclude_dirs=["top_selection"] prevents previously-promoted bundles
#    from being included in the index, keeping the sweep results clean.
index_path = build_index(
    str(backtest_results_dir),
    show_progress=True,
    incremental=True,
    exclude_dirs=["top_selection"],
)
print(f"Index written to: {index_path}")

# 2. Rank using weighted multi-metric scoring with a BALANCED focus.
top = rank_index(
    str(backtest_results_dir),
    focus=BacktestEvaluationFocus.BALANCED,
    where="summary_number_of_trades_closed > 5",
    engine="vector",
    study=study,
    limit=20,
)
print(f"\nTop {len(top)} strategies (BALANCED focus, vector engine):\n")
print(format_table(top))

# 3. Promote the top-N bundles into a fresh ``top_selection/`` folder
#    for notebook 04 to consume. ``clear_dest=True`` wipes the folder
#    first so it always reflects exactly this run's selection.
top_selection_path = backtest_results_dir / "top_selection"
result = promote_backtests(
    str(backtest_results_dir),
    keep=top,
    dest_dir=str(top_selection_path),
    mode="copy",
    clear_dest=True,
    flatten=True,
    show_progress=True,
)
print(
    f"\nPromoted {result['promoted']} winning bundles → {result['dest_dir']}"
)

## Analysis & Reporting

In [ ]:
from pathlib import Path
from investing_algorithm_framework import (
    BacktestEvaluationFocus, build_index, rank_index, LocalDirStore,
)

backtest_results_dir = Path.cwd().parent / "backtest_results"
top_selection_path = backtest_results_dir / "top_selection"

# 1. Build (or refresh) the Tier-1 SQLite index for top selection
top_selection_index_path = build_index(
    str(top_selection_path),
    show_progress=True,
    incremental=True,
)

# 2. Re-rank the top selection with the same BALANCED focus.
#    Scoping to (engine="vector", study=study)
#    guarantees one row per bundle, so the result preserves rank
#    order with no duplication.
top = rank_index(
    str(top_selection_path),
    focus=BacktestEvaluationFocus.BALANCED,
    engine="vector",
    study=study,
)

# 3. Materialise the ranked winners into ``Backtest`` objects so the
#    downstream metric-table helpers (which require ``Backtest``
#    instances, not raw SQLite index rows) can read per-run and
#    summary metrics.
store = LocalDirStore(str(top_selection_path))
backtests = [store.open(row["bundle_path"]) for row in top]
print(f"Loaded {len(backtests)} backtests from top selection for analysis")

In [ ]:
from investing_algorithm_framework import (
    show_backtest_summaries, show_backtest_runs, DEFAULT_TRADE_METRIC_COLUMNS
)

# NOTE: the show_* helpers operate on materialised ``Backtest`` objects
# (the ``backtests`` list opened via ``store.open`` in the previous
# cell). ``top`` from ``rank_index`` is a list of raw SQLite index
# rows (dicts) and is only used for ranking / pruning.
#
# We pass ``study=study`` so the helpers resolve the correct vector /
# event slot in multi-study bundles via
# ``Backtest.get_summary(engine, study=...)`` and
# ``Backtest.get_runs(engine, study=...)``. Without it, multi-study
# bundles would raise under the framework's default-study rule.

show_backtest_summaries(
    backtests,
    engine="vector",
    study=study,
    sort_by="cagr",
)

show_backtest_runs(
    backtests,
    engine="vector",
    study=study,
    run=rolling_backtest_windows,
    page_size=25,
)

show_backtest_summaries(
    backtests,
    engine="vector",
    study=study,
    columns=DEFAULT_TRADE_METRIC_COLUMNS,
    sort_by="profit_factor",
)

show_backtest_runs(
    backtests,
    engine="vector",
    study=study,
    columns=DEFAULT_TRADE_METRIC_COLUMNS,
    run=rolling_backtest_windows,
    page_size=25,
)


## Displaying the results in a HTML Report

In [ ]:
from investing_algorithm_framework import BacktestReport

report = BacktestReport(backtests=backtests, study=study)
report.show(browser=True)